In [2]:
# 문서 객체와 기본 키워드 검색기 모듈을 가져옵니다.
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# 여러 주제가 혼합된 실습용 사내 규정 문서를 정의합니다.
raw_documents = [
    Document(
        page_content="복지 제도 안내: 사내 복지 포인트는 매년 1월 1일에 100만 포인트가 지급됩니다. "
                     "포인트는 자기계발 및 문화생활에 사용 가능하며, 미사용 잔여 포인트는 당해 연도 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다."
    ),
    Document(
        page_content="휴가 규정 안내: 연차 유급휴가는 1년간 80% 이상 출근 시 15일이 부여됩니다. "
                     "경조사 휴가는 사유 발생일로부터 사용해야 하며, 미사용 연차는 연말에 수당으로 정산 지급됩니다."
    ),
    Document(
        page_content="근무 환경 가이드: 사무실 내 회의실 예약은 사내 인트라넷 시스템을 통해 최소 1시간 전에 신청해야 합니다. "
                     "점심시간은 12시 30분부터 13시 30분까지이며 부서별 자율 운영이 가능합니다."
    )
]

# 공백 기준 단순 토크나이저 함수를 정의합니다.
def custom_tokenizer(text: str) -> list[str]:
    # 문장을 띄어쓰기 단위로 분리하여 단어 리스트를 반환합니다.
    return text.split()

# 1단계 기본 검색기(BM25)를 생성하고 검색 개수를 1개로 설정합니다.
base_retriever = BM25Retriever.from_documents(
    documents=raw_documents,
    preprocess_func=custom_tokenizer
)
base_retriever.k = 2

In [4]:
from langchain.chat_models import init_chat_model
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low"
)

compressor = LLMChainExtractor.from_llm(model)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [5]:
query = "사내 복지 포인트 남으면 다음 해로 넘겨서 쓸 수 있나요?"

print("=== BM25 ===")
bm25_docs = base_retriever.invoke(query)
for i, doc in enumerate(bm25_docs, 1):
    print(f"순서: {i}, Doc:{doc.page_content}")

print("=== Compression ===")
com_docs = compression_retriever.invoke(query)
for i, doc in enumerate(com_docs, 1):
    print(f"순서: {i}, Doc:{doc.page_content}")

=== BM25 ===
순서: 1, Doc:복지 제도 안내: 사내 복지 포인트는 매년 1월 1일에 100만 포인트가 지급됩니다. 포인트는 자기계발 및 문화생활에 사용 가능하며, 미사용 잔여 포인트는 당해 연도 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다.
순서: 2, Doc:근무 환경 가이드: 사무실 내 회의실 예약은 사내 인트라넷 시스템을 통해 최소 1시간 전에 신청해야 합니다. 점심시간은 12시 30분부터 13시 30분까지이며 부서별 자율 운영이 가능합니다.
=== Compression ===
순서: 1, Doc:복지 제도 안내: 사내 복지 포인트는 매년 1월 1일에 100만 포인트가 지급됩니다. 포인트는 자기계발 및 문화생활에 사용 가능하며, 미사용 잔여 포인트는 당해 연도 12월 31일에 전액 소멸되어 다음 해로 이월되지 않습니다.
